# 06 — Model Evaluation & Threshold Analysis

In [4]:
from pathlib import Path
import sys, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().resolve()
if not (ROOT/"src").exists(): ROOT = ROOT.parent
sys.path.append(str(ROOT))
DATA = ROOT/"data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
assert DATA.exists(), f"Download the dataset and place it at {DATA}"

In [5]:
from sklearn.model_selection import train_test_split
from src.data_cleaning import clean_data, prepare_target
from src.feature_engineering import add_business_features
from src.train_model import build_models, train_pipeline
from src.evaluate_model import threshold_table

In [6]:
df=add_business_features(prepare_target(clean_data(pd.read_csv(DATA))))
X=df.drop(columns=['Churn','customerID']); y=df.Churn
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
trained={n:train_pipeline(Xtr,ytr,e) for n,e in build_models().items()}
# Candidate: prioritize recall and PR-AUC, then inspect threshold trade-offs.
for n,m in trained.items():
    print('\n',n); display(threshold_table(m,Xte,yte).round(4))


 logistic_regression


,threshold,precision,recall,f1,target_rate
0,0.20,0.3900,0.9626,0.5551,0.6551
1,0.25,0.4074,0.9412,0.5687,0.6132
2,0.30,0.4277,0.9251,0.5850,0.5742
3,0.35,0.4484,0.9064,0.6000,0.5366
4,0.40,0.4713,0.8770,0.6131,0.4940
5,0.45,0.4867,0.8316,0.6140,0.4535
6,0.50,0.4992,0.7968,0.6138,0.4237
7,0.55,0.5390,0.7567,0.6296,0.3726
8,0.60,0.5528,0.7139,0.6231,0.3428
9,0.65,0.5745,0.6497,0.6098,0.3002



 random_forest


,threshold,precision,recall,f1,target_rate
0,0.20,0.4145,0.9144,0.5705,0.5855
1,0.25,0.4456,0.8877,0.5934,0.5287
2,0.30,0.4785,0.8610,0.6151,0.4776
3,0.35,0.4927,0.8075,0.6120,0.4351
4,0.40,0.5275,0.7941,0.6339,0.3996
5,0.45,0.5377,0.7246,0.6173,0.3577
6,0.50,0.5711,0.6872,0.6238,0.3194
7,0.55,0.5783,0.6123,0.5948,0.2811
8,0.60,0.6138,0.5481,0.5791,0.2370
9,0.65,0.6512,0.4893,0.5588,0.1994



 xgboost


,threshold,precision,recall,f1,target_rate
0,0.20,0.4830,0.8369,0.6125,0.4599
1,0.25,0.5087,0.7781,0.6152,0.4060
2,0.30,0.5355,0.7460,0.6235,0.3698
3,0.35,0.5620,0.7032,0.6247,0.3322
4,0.40,0.5738,0.6444,0.6071,0.2981
5,0.45,0.6082,0.5936,0.6008,0.2590
6,0.50,0.6632,0.5160,0.5805,0.2065
7,0.55,0.6875,0.4706,0.5587,0.1817
8,0.60,0.6967,0.3930,0.5026,0.1498
9,0.65,0.7545,0.3369,0.4658,0.1185


Select the final threshold using outreach capacity and the cost of false negatives vs false positives. Record the decision in `reports/model_card.md`.